In [1]:
from utils.load_ml_datasets import load_ml100k
from utils.constants import CONTEXTS_PER_BATCH, ITEM_ID_COLUMN
from utils.mab2rec_wrappers import LinUCBMab2RecWrapper, LinGreedyMab2RecWrapper, LinTSMab2RecWrapper
from utils.gobrec_wrappers import LinUCBGobrecWrapperCPU, LinGreedyGobrecWrapperCPU, LinTSGobrecWrapperCPU, \
                                  LinUCBGobrecWrapperGPU, LinGreedyGobrecWrapperGPU, LinTSGobrecWrapperGPU
from utils.BaseWrapper import BaseWrapper
import time

LinUCBGobrecWrapperGPU() # Just to init GPU

import pandas as pd
import numpy as np

In [2]:
DATASET_SIZE = 100_000 # Max 100_000

In [3]:
def ndcg(recs: np.ndarray, item_ids: np.ndarray):
    idcg = len(item_ids)
    
    recs_ranks = np.where(item_ids.reshape(-1, 1) == recs)[1]
    dcg = np.sum(1 / np.log2(recs_ranks + 2))

    return dcg / idcg

In [ ]:

def execute_not_incremental_experiment(wrapper: BaseWrapper, interactions_df: pd.DataFrame, contexts: np.ndarray):
    num_items = interactions_df[ITEM_ID_COLUMN].nunique()

    split_index = int(len(interactions_df) * 0.5)

    train_df = interactions_df.copy()[:split_index]

    train_contexts = contexts[:split_index]
    test_contexts = contexts[split_index:]
    
    for start in range(0, len(train_contexts), CONTEXTS_PER_BATCH):
        if start == 0:
            wrapper.fit(train_df[start:start+CONTEXTS_PER_BATCH], train_contexts[start:start+CONTEXTS_PER_BATCH], num_items=num_items)
        else:
            wrapper.partial_fit(train_df[start:start+CONTEXTS_PER_BATCH], train_contexts[start:start+CONTEXTS_PER_BATCH])

    results = []
    for start in range(0, len(test_contexts), CONTEXTS_PER_BATCH):
        result = wrapper.recommend(test_contexts[start:start+CONTEXTS_PER_BATCH])
        results.append(result)
    
    return results

def compare_algorithms(interactions_df: pd.DataFrame, contexts: np.ndarray, mab2rec_algo: BaseWrapper, gobrec_algo: BaseWrapper, scores_tolerance: float = 1e-5):
    start_time_mab2rec = time.time()
    results_mab2rec = execute_not_incremental_experiment(mab2rec_algo, interactions_df, contexts.copy())
    elapsed_time_mab2rec = time.time() - start_time_mab2rec

    start_time_gobrec = time.time()
    results_gobrec = execute_not_incremental_experiment(gobrec_algo, interactions_df, contexts.copy())
    elapsed_time_gobrec = time.time() - start_time_gobrec

    recs_mab2rec = np.concatenate([np.array(batch_results[0]) for batch_results in results_mab2rec])
    recs_gobrec = np.concatenate([batch_results[0] for batch_results in results_gobrec])
    
    scores_mab2rec = np.concatenate([np.array(batch_results[1]) for batch_results in results_mab2rec])
    scores_gobrec = np.concatenate([batch_results[1] for batch_results in results_gobrec])

    print(f"Elapsed time for MAB2Rec:\n {elapsed_time_mab2rec:.4f} seconds")
    print(f"Elapsed time for Gobrec:\n {elapsed_time_gobrec:.4f} seconds (x {elapsed_time_mab2rec / elapsed_time_gobrec:.2f} faster)\n")

    different_recs_number = np.sum(recs_mab2rec != recs_gobrec)
    print(f"Number of different recommendations:\n {different_recs_number} out of {recs_mab2rec.shape[0] * recs_mab2rec.shape[1]} ({different_recs_number / (recs_mab2rec.shape[0] * recs_mab2rec.shape[1]) * 100:.2f}%):")
  
    scores_difference = np.sum(~np.isclose(scores_mab2rec, scores_gobrec, atol=scores_tolerance))
    print(f"Number of different scores:\n {scores_difference} out of {scores_mab2rec.shape[0] * scores_mab2rec.shape[1]} ({scores_difference / (scores_mab2rec.shape[0] * scores_mab2rec.shape[1]) * 100:.2f})%\n")

    eval_item_ids = interactions_df[ITEM_ID_COLUMN].values[len(interactions_df) // 2:]
    ndcg_mab2rec = ndcg(recs_mab2rec, eval_item_ids)
    ndcg_gobrec = ndcg(recs_gobrec, eval_item_ids)
    print(f"NDCG for MAB2Rec: {ndcg_mab2rec:.4f}")
    print(f"NDCG for Gobrec: {ndcg_gobrec:.4f}")
    print(f"NDCG difference: {abs(ndcg_mab2rec - ndcg_gobrec):.4f} ({abs(ndcg_mab2rec - ndcg_gobrec) / ndcg_mab2rec * 100:.2f}%)")

In [5]:
interactions_df_ml100k, contexts_ml100k = load_ml100k()
interactions_df_ml100k = interactions_df_ml100k[:DATASET_SIZE]
contexts_ml100k = contexts_ml100k[:DATASET_SIZE]

contexts_ml100k = contexts_ml100k[interactions_df_ml100k['rating'] >= 4]
interactions_df_ml100k = interactions_df_ml100k[interactions_df_ml100k['rating'] >= 4]

interactions_df_ml100k = interactions_df_ml100k[~np.all(contexts_ml100k == 0, axis=1)]
contexts_ml100k = contexts_ml100k[~np.all(contexts_ml100k == 0, axis=1)]

## LinUCB mab2rec vs GOBRec CPU

In [6]:
linucb_mab2rec = LinUCBMab2RecWrapper()
linucb_gobrec_cpu = LinUCBGobrecWrapperCPU()

compare_algorithms(interactions_df_ml100k, contexts_ml100k, linucb_mab2rec, linucb_gobrec_cpu)

Elapsed time for MAB2Rec:
 314.9481 seconds
Elapsed time for Gobrec:
 1.6700 seconds (x 188.59 faster)

Number of different recommendations:
 0 out of 547900 (0.00%):
Number of different scores:
 0 out of 547900 (0.00)%

NDCG for MAB2Rec: 0.0251
NDCG for Gobrec: 0.0251
NDCG difference: 0.0000 (0.00%)


## LinUCB mab2rec vs GOBRec GPU

In [7]:
linucb_mab2rec = LinUCBMab2RecWrapper()
linucb_gobrec_cpu = LinUCBGobrecWrapperGPU()

compare_algorithms(interactions_df_ml100k, contexts_ml100k, linucb_mab2rec, linucb_gobrec_cpu)

Elapsed time for MAB2Rec:
 314.4243 seconds
Elapsed time for Gobrec:
 0.7966 seconds (x 394.69 faster)

Number of different recommendations:
 0 out of 547900 (0.00%):
Number of different scores:
 0 out of 547900 (0.00)%

NDCG for MAB2Rec: 0.0251
NDCG for Gobrec: 0.0251
NDCG difference: 0.0000 (0.00%)


## LinGreedy mab2rec vs GOBRec CPU

In [8]:
lingreedy_mab2rec = LinGreedyMab2RecWrapper()
lingreedy_gobrec_cpu = LinGreedyGobrecWrapperCPU()

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lingreedy_mab2rec, lingreedy_gobrec_cpu)

Elapsed time for MAB2Rec:
 312.5727 seconds
Elapsed time for Gobrec:
 0.2435 seconds (x 1283.90 faster)

Number of different recommendations:
 103253 out of 547900 (18.85%):
Number of different scores:
 103316 out of 547900 (18.86)%

NDCG for MAB2Rec: 0.0270
NDCG for Gobrec: 0.0273
NDCG difference: 0.0002 (0.88%)


In [9]:
lingreedy_mab2rec = LinGreedyMab2RecWrapper(epsilon=0)
lingreedy_gobrec_cpu = LinGreedyGobrecWrapperCPU(epsilon=0)

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lingreedy_mab2rec, lingreedy_gobrec_cpu)

Elapsed time for MAB2Rec:
 310.9288 seconds
Elapsed time for Gobrec:
 0.2486 seconds (x 1250.91 faster)

Number of different recommendations:
 0 out of 547900 (0.00%):
Number of different scores:
 0 out of 547900 (0.00)%

NDCG for MAB2Rec: 0.0296
NDCG for Gobrec: 0.0296
NDCG difference: 0.0000 (0.00%)


## LinGreedy mab2rec vs GOBRec GPU

In [10]:
lingreedy_mab2rec = LinGreedyMab2RecWrapper()
lingreedy_gobrec_cpu = LinGreedyGobrecWrapperGPU()

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lingreedy_mab2rec, lingreedy_gobrec_cpu)

Elapsed time for MAB2Rec:
 342.1963 seconds
Elapsed time for Gobrec:
 0.2054 seconds (x 1666.36 faster)

Number of different recommendations:
 102846 out of 547900 (18.77%):
Number of different scores:
 102889 out of 547900 (18.78)%

NDCG for MAB2Rec: 0.0270
NDCG for Gobrec: 0.0275
NDCG difference: 0.0005 (1.84%)


In [11]:
lingreedy_mab2rec = LinGreedyMab2RecWrapper(epsilon=0)
lingreedy_gobrec_cpu = LinGreedyGobrecWrapperGPU(epsilon=0)

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lingreedy_mab2rec, lingreedy_gobrec_cpu)

Elapsed time for MAB2Rec:
 322.8408 seconds
Elapsed time for Gobrec:
 0.1351 seconds (x 2389.46 faster)

Number of different recommendations:
 0 out of 547900 (0.00%):
Number of different scores:
 0 out of 547900 (0.00)%

NDCG for MAB2Rec: 0.0296
NDCG for Gobrec: 0.0296
NDCG difference: 0.0000 (0.00%)


## LinTS mab2rec vs GOBRec CPU

In [12]:
lints_mab2rec = LinTSMab2RecWrapper()
lints_gobrec_cpu = LinTSGobrecWrapperCPU()

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lints_mab2rec, lints_gobrec_cpu)

Elapsed time for MAB2Rec:
 361.6578 seconds
Elapsed time for Gobrec:
 1.6481 seconds (x 219.44 faster)

Number of different recommendations:
 412442 out of 547900 (75.28%):
Number of different scores:
 547343 out of 547900 (99.90)%

NDCG for MAB2Rec: 0.0295
NDCG for Gobrec: 0.0297
NDCG difference: 0.0002 (0.76%)


In [13]:
lints_mab2rec = LinTSMab2RecWrapper(alpha=0.01)
lints_gobrec_cpu = LinTSGobrecWrapperCPU(alpha=0.01)

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lints_mab2rec, lints_gobrec_cpu)

Elapsed time for MAB2Rec:
 354.3548 seconds
Elapsed time for Gobrec:
 1.6367 seconds (x 216.50 faster)

Number of different recommendations:
 120636 out of 547900 (22.02%):
Number of different scores:
 542572 out of 547900 (99.03)%

NDCG for MAB2Rec: 0.0297
NDCG for Gobrec: 0.0296
NDCG difference: 0.0001 (0.26%)


## LinTS mab2rec vs GOBRec GPU

In [14]:
lints_mab2rec = LinTSMab2RecWrapper()
lints_gobrec_cpu = LinTSGobrecWrapperGPU()

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lints_mab2rec, lints_gobrec_cpu)

Elapsed time for MAB2Rec:
 356.0067 seconds
Elapsed time for Gobrec:
 0.1723 seconds (x 2066.01 faster)

Number of different recommendations:
 411848 out of 547900 (75.17%):
Number of different scores:
 547304 out of 547900 (99.89)%

NDCG for MAB2Rec: 0.0295
NDCG for Gobrec: 0.0297
NDCG difference: 0.0002 (0.67%)


In [15]:
lints_mab2rec = LinTSMab2RecWrapper(alpha=0.01)
lints_gobrec_cpu = LinTSGobrecWrapperGPU(alpha=0.01)

compare_algorithms(interactions_df_ml100k, contexts_ml100k, lints_mab2rec, lints_gobrec_cpu)

Elapsed time for MAB2Rec:
 348.0062 seconds
Elapsed time for Gobrec:
 0.1648 seconds (x 2111.82 faster)

Number of different recommendations:
 120357 out of 547900 (21.97%):
Number of different scores:
 542586 out of 547900 (99.03)%

NDCG for MAB2Rec: 0.0297
NDCG for Gobrec: 0.0297
NDCG difference: 0.0000 (0.03%)
